### **Customer Churn Prediction and Financial Customer Lifetime Value (CLV) Forecasting in Telecommunications**


>**Organization:** MTN Rwanda\
>**Department:** CBU / Business Intelligence\
>**Project Type:** Applied Data Science and Machine Learning\
>**Primary Tools:** Python, pandas, NumPy, scikit-learn, Matplotlib/Seaborn, and Streamlit
>

### **Project Background**
Telecommunications companies serve customers with different usage patterns, spending habits, product preferences, and levels of engagement. Some customers generate consistent revenue over time, while others gradually reduce their spending or stop generating revenue altogether.

Understanding customer churn and predicting future customer value can help telecom operators identify subscribers who may become inactive, estimate the revenue associated with retaining valuable customers, and make better decisions about retention campaigns.

Traditional reporting can show historical customer revenue and inactivity. However, Machine Learning can help estimate the likelihood of future churn, while predictive financial analysis can estimate the revenue a customer may generate over a future period.

This project develops an analytical solution that combines customer behavior analysis, churn prediction, and customer lifetime value forecasting to support data-driven retention decisions.


#### **Project Objectives**
The main objective is to develop a data science solution that predicts customer churn risk and estimates the future financial value of telecom subscribers, helping the business prioritize retention activities.

##### **Specific objectives**

1. Analyze customer behavior and identify patterns associated with declining activity and churn.
2. Develop a Machine Learning model to predict which customers are likely to churn within a defined future period.
3. Build a model to estimate future customer revenue over a defined forecasting horizon.
4. Combine churn probability and predicted customer value to rank customers for potential retention activities.
5. Evaluate model performance using appropriate statistical and Machine Learning metrics.
6. Translate model outputs into actionable business insights and recommendations.


### Imports
Import the Python packages and libraries used in the project.

In [ ]:
# Import necessary packages for data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import scikit-learn for machine learning and predictive data analysis
from sklearn.model_selection import  train_test_split, ParameterGrid, StratifiedKFold, KFold, cross_val_score, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing  import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error, r2_score, brier_score_loss

import math
import matplotlib.dates as mdates

### Exploratory Data Analysis
Investigating, summarizing, and visualizing data to uncover its main characteristics, hidden patterns, distributions, and relationships.

In [ ]:
# Load the dataset
file_path = "Project_Dataset.csv"
df = pd.read_csv(file_path)

In [ ]:
# print the first 5 rows
df.head()

In [ ]:
# (number of rows, number of columns)
df.shape

In [ ]:
df.info()

In [ ]:
# number of missing values per column
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# statistical summary of each column
df.describe()

In [ ]:
# convert customer_id to category to save memory and speed up performance
df["customer_id"] = df["customer_id"].astype("category")

# convert date to appropriate datetime format
if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")

# sort dataset by customer_id and date
df = df.sort_values(["customer_id", "date"]).reset_index(drop=True)
display(df.head())

# turn the revenue (negative balance deductions) positive
revenue_cols = ["data_revenue", "voice_revenue", "sms_revenue"]
df[revenue_cols] = df[revenue_cols].abs()

# create total_revenue column
df["total_revenue"] = df[revenue_cols].sum(axis=1)

In [ ]:
# date range
start, end = df["date"].min(), df["date"].max()
n_days = (end - start).days + 1
print(f"From {start.date()} to {end.date()} -> {n_days} days")

# missing dates
all_dates = pd.date_range(start, end, freq="D")
missing_dates = all_dates.difference(df["date"].unique())
print(f"Missing dates:{len(missing_dates)}\n")
if len(missing_dates):
    print(missing_dates)

# rows per date
# how many customers have a row on that date
rows_per_day = df.groupby("date").size()
print(rows_per_day.describe())

fig, ax = plt.subplots(figsize=(12, 4))
rows_per_day.plot(ax=ax, x_compat=True)
# ticks at month boundaries, labels in the middle of each month
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(plt.NullFormatter())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonthday=15))
ax.xaxis.set_minor_formatter(mdates.DateFormatter("%b-%Y"))
ax.tick_params(axis="x", which="minor", length=0)
ax.set_xlim(start, end)
plt.setp(ax.get_xticklabels(minor=True), rotation=45, ha="right")
ax.set_title("Number of Rows per Day")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.show()


In [ ]:
# number of unique customer_id in the dataset
n_customers = df["customer_id"].nunique()
rows_per_customer = df.groupby("customer_id", observed=True).size()

# print the number of unique customers
print(f"Customers: {n_customers}")
print(f"Expected rows if complete: {n_customers * n_days:,} | Actual: {len(df):,}\n")
# summary of days per customer
print(rows_per_customer.describe())

In [ ]:
# Plot activity over time

daily = df.groupby("date")[["total_revenue", "data_traffic", "voice_traffic", "recharge_amount"]].sum()
axes = daily.plot(subplots=True, figsize=(12, 8), title="Daily Totals across All Customers", x_compat=True)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))

plt.setp(axes[-1].get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# customer-level dataset: one row per customer, total activity over the period
base_cols = ["data_revenue", "voice_revenue", "sms_revenue", "data_traffic",
             "voice_traffic", "recharge_amount", "recharge_count"]
customer_df = df.groupby("customer_id", observed=True)[base_cols].sum()
display(customer_df.head())

In [ ]:
# Distribution summary includes skewness and share of customers with zero activity per column
dist_summary = pd.DataFrame({
    "skewness": customer_df.skew(),
    "zero_share_%": (customer_df == 0).mean() * 100,
    "median": customer_df.median(),
    "mean": customer_df.mean(),
    "max": customer_df.max(),
}).round(2).sort_values("skewness", ascending=False)

display(dist_summary)

In [ ]:
zero_data_revenue = customer_df["data_revenue"] == 0
print("Customers with zero data revenue:", zero_data_revenue.sum())
print("... of which have zero data traffic:", (zero_data_revenue & (customer_df["data_traffic"] == 0)).sum())
print("... of which have data traffic > 0:", (zero_data_revenue & (customer_df["data_traffic"] > 0)).sum())

In [ ]:
spend_gap = customer_df["recharge_amount"] - customer_df[["data_revenue", "voice_revenue", "sms_revenue"]].sum(axis=1)
print(spend_gap.describe())
print("Customers where recharge > recorded revenue:", (spend_gap > 0).sum())

In [ ]:
# check extremes to see if they are real customers or data errors
display(spend_gap.nsmallest(5))
display(spend_gap.nlargest(5))

In [ ]:
# how many in each gap group
print("Gap > 0:", (spend_gap > 0).sum())
print("Gap = 0:", (spend_gap == 0).sum())
print("Gap < 0:", (spend_gap < 0).sum())

# are the zero-gap customers fully inactive?
zero_gap = spend_gap == 0
print("Zero-gap customers with no recharge and no revenue:",
      (zero_gap & (customer_df["recharge_amount"] == 0)).sum())

# do customers who use data without paying data revenue have a positive gap?
bundle_users = (customer_df["data_traffic"] > 0) & (customer_df["data_revenue"] == 0)
print("Median gap, data users with zero data revenue:", spend_gap[bundle_users].median())
print("Median gap, everyone else:", spend_gap[~bundle_users].median())

In [ ]:
top_gap_id = spend_gap.idxmax()
print("Customer:", top_gap_id)
print(f"Spend gap: {spend_gap[top_gap_id]:,.0f}\n")
display(customer_df.loc[[top_gap_id]].T.style.format("{:,.2f}"))

one = df[df["customer_id"] == top_gap_id]
print("Days with a recharge:", (one["recharge_amount"] > 0).sum())
print("Days with revenue:", (one["total_revenue"] > 0).sum())
display(one.loc[one["recharge_amount"] > 0, ["date", "recharge_amount", "recharge_count", "total_revenue", "voice_traffic", "data_traffic"]].head(20))

In [ ]:
bundle_users = (customer_df["data_traffic"] > 0) & (customer_df["data_revenue"] == 0)
print("Of these, customers with voice revenue > 0:", (bundle_users & (customer_df["voice_revenue"] > 0)).sum())
print("Of these, customers with zero total revenue:",
      (bundle_users & (customer_df[["data_revenue", "voice_revenue", "sms_revenue"]].sum(axis=1) == 0)).sum())


In [ ]:
# histogram (log scale)
sns.set_style("darkgrid")
cols = customer_df.columns
n_rows = math.ceil(len(cols) / 2)

fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 3))
for ax, feature in zip(axes.flat, cols):
    sns.histplot(customer_df[feature] + 1, log_scale=True, bins=50, ax=ax)
    ax.set_title(f"{feature} | Skewness: {customer_df[feature].skew():.2f}")
    ax.set_xlabel(f"{feature} (+1, log scale)")
for ax in axes.flat[len(cols):]:
    ax.set_visible(False)

plt.suptitle("Distribution of Total Activity per Customer", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# box plots on a log scale
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=customer_df + 1, orient="h", log_scale=True, ax=ax)
ax.set_title("Customer Totals (log scale, +1)")
plt.tight_layout()
plt.show()

# count high outliers per column with the IQR rule,
# using only customers with non-zero values (zeros distort the quartiles)
log_df = np.log1p(customer_df)
outliers = pd.DataFrame(False, index=customer_df.index, columns=customer_df.columns)

for col in customer_df.columns:
    nonzero = log_df.loc[customer_df[col] > 0, col]
    q1, q3 = nonzero.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    outliers[col] = log_df[col] > upper

print("High outlier customers per column:")
print(outliers.sum())
print(f"\nCustomers who are an outlier in at least one column: {outliers.any(axis=1).sum()}")

# top 5 customers by recharge
display(customer_df.nlargest(5, "recharge_amount"))

In [ ]:
# create a correlation matrix for numeric columns only, without date
corr_df = df[base_cols]
corr_matrix  = corr_df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            vmin=-1, vmax=1, center=0,
            square=True,
            linewidth=0.5,
            ax=ax
)
ax.set_title("Spearman Correlation of Daily Customer Activity")
plt.tight_layout()
plt.show()

Row-level correlations are weak because activity is sparse and bundle purchases are separated in time from usage, so analysis is done at the customer level.

In [ ]:
# customer-level correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    customer_df.corr(method="spearman"),
    annot=True,
    cmap="coolwarm",
    vmin=-1, vmax=1, center=0,
    square=True,
    linewidth=0.5,
    ax=ax
)
ax.set_title("Spearman Correlation of Total Activity per Customer")
plt.tight_layout()
plt.show()

In [ ]:
active_customers = customer_df[customer_df[revenue_cols].sum(axis=1) > 0]
print(f"Active customers (revenue > 0 during the period): {len(active_customers)}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    active_customers.corr(method="spearman"),
    annot=True,
    cmap="coolwarm",
    vmin=-1, vmax=1, center=0,
    square=True,
    linewidth=0.5,
)
plt.title("Spearman Correlation (Customers with Revenue)")
plt.show()

In [ ]:
# an active day is one where a customer generated revenue
df["active"] = df["total_revenue"] > 0
display(df.head())

In [ ]:
# inactivity patterns
df["block"] = df.groupby("customer_id", observed=True)["active"].cumsum()

runs = (
    df[~df["active"]]
    .groupby(["customer_id", "block"], observed=True)["date"]
    .agg(start="min", end="max", length="size")
)
runs = runs[runs.index.get_level_values("block") > 0]
runs["returned"] = runs["end"] < end

print(runs["length"].describe())
print("\n")

# share of inactive streaks that ended with the customer paying again, for each possible churn threshold
for t in [7, 14, 30, 60, 90, 120, 150]:
    hit = runs[runs["length"] >= t]
    print(f"{t:>3} days inactive: {len(hit):>5} streaks | came back: {hit['returned'].mean():.1%}")

In [ ]:
# recharge frequency
recharges = df.loc[df["recharge_amount"] > 0, ["customer_id", "date"]].copy()
recharges["gap_days"] = recharges.groupby("customer_id", observed=True)["date"].diff().dt.days

median_gap = recharges.groupby("customer_id", observed=True)["gap_days"].median()
print(median_gap.describe())
sns.histplot(median_gap.dropna(), bins=100)
plt.title("Typical Days Between Recharges per Customer")
plt.show()

In [ ]:
# revenue changes over time
monthly = (
    df.groupby(["customer_id", df["date"].dt.to_period("M")], observed=True)["total_revenue"]
    .sum()
    .unstack()
) # one row per customer, one column per month

# active customers and median revenue per month
active_per_month = (monthly > 0).sum()
median_per_month = monthly[monthly > 0].median()
month_labels = monthly.columns.strftime("%b-%Y") # e.g. Sep-2025

fig, axes = plt.subplots(1, 2, figsize=(14, 4)) # 1 row x 2 columns of plots
axes[0].plot(month_labels, active_per_month.values, marker="o")
axes[0].set_title("Customers with Revenue per Month")
axes[1].plot(month_labels, median_per_month.values, marker="o")
axes[1].set_title("Median Revenue per Paying Customer per Month")
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

# compare each customer's first 3 months with their last 3 months
first_3 = monthly.iloc[:, :3].mean(axis=1)
last_3 = monthly.iloc[:, -3:].mean(axis=1)
middle = monthly.iloc[:, 3:-3].mean(axis=1)

change =  (last_3 - first_3) / (first_3 + 1)
trend = pd.cut(
    change,
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["declining", "stable", "growing"],
)

trend = trend.cat.add_categories(["inactive all year", "active mid-year only", "late joiner"])
both_ends_zero = (first_3 == 0) & (last_3 == 0)
trend[both_ends_zero & (middle == 0)] = "inactive all year"
trend[both_ends_zero & (middle > 0)] = "active mid-year only"
trend[first_3.isna()] = "late joiner"

print(trend.value_counts())


In [ ]:
"""
A customer is churned when they have no revenue-generating activity for 90 consecutive days.
The last 90 days of data form the outcome window; everything before is the observation window, used to build features.
Customers with no revenue in the 90 days before the cutoff are excluded, since they had already churned.
"""

# Split data into observation window (features) and outcomes
cutoff = end - pd.Timedelta(days=90)
observation = df[df["date"] <= cutoff]
outcome = df[df["date"] > cutoff]

# keep only customers active at the cutoff
recent_observation = observation[observation["date"] > cutoff - pd.Timedelta(days=90)]
active_at_cutoff = recent_observation.groupby("customer_id", observed=True)["total_revenue"].sum() > 0
eligible = active_at_cutoff[active_at_cutoff].index

# Churn label, explained in the Churn Model section
# churned customers
outcome_revenue = outcome.groupby("customer_id", observed=True)["total_revenue"].sum()
churn = (outcome_revenue.loc[eligible] == 0).astype(int).rename("churned")

print(f"Observation window: {start.date()} to {cutoff.date()}")
print(f"Outcome window: {(cutoff + pd.Timedelta(days=1)).date()} to {end.date()}")
print(f"Already inactive at cutoff (excluded): {(~active_at_cutoff).sum()}")
print(f"Customers in churn analysis: {len(churn)}")
print(churn.value_counts())
print(f"Churn rate: {churn.mean():.1%}")

In [ ]:
excluded = active_at_cutoff[~active_at_cutoff].index
reactivated = outcome_revenue.loc[excluded] > 0

print(f"Dormant at cutoff: {len(excluded)}")
print(f"Generated revenue in the next 90 days: {reactivated.sum()} ({reactivated.mean():.1%})")


In [ ]:
excluded = active_at_cutoff[~active_at_cutoff].index
ever_paid = observation.groupby("customer_id", observed=True)["total_revenue"].sum().loc[excluded] > 0
reactivated = outcome_revenue.loc[excluded] > 0

for label, group in [("Paid before, then went quiet", ever_paid), ("Never paid in observation", ~ever_paid)]:
    print(f"{label}: {group.sum()} customers | came back: {reactivated[group].mean():.1%}")


### Feature Engineering
Selecting, creating and transforming raw data into meaningful inputs that improve model accuracy and performance.

In [ ]:
""" features are built only from the observation window,
so the model never sees data from the period it is predicting. """

# observation window, eligible customers only
win = observation[observation["customer_id"].isin(eligible)]

# empty table: one row per eligible customer, features added column by column
features = pd.DataFrame(index=eligible)

# helper functions
def window(days_from, days_to=0):
    """
    rows between 'days_from' and 'days_to' before cutoff
    window(30) # last 30 days
    window(60, 30) # days 31-60 before cutoff
    """
    return win[(win["date"] > cutoff - pd.Timedelta(days=days_from)) & 
               (win["date"] <= cutoff - pd.Timedelta(days=days_to))]

def per_customer(data, col, how="sum"):
    """
    aggregate one column per customer
    customers with no rows get 0
    """
    return (data.groupby("customer_id", observed=True)[col]
            .agg(how)
            .reindex(eligible, fill_value=0))

def pct_change(recent,  previous):
    """ relative change """
    return (recent - previous) / (previous + 1)

In [ ]:
""" revenue features """

# tenure: days from first appearance in the data to the cutoff
first_seen = win.groupby("customer_id", observed=True)["date"].min()
features["tenure_days"] = (cutoff - first_seen).dt.days + 1

# total revenue over the last 30, 60, and 90 days
for d in [30, 60, 90]:
    features[f"revenue_{d}d"] = per_customer(window(d), "total_revenue")

# average daily revenue and comparison between usual and recent (30 days) revenue
features["avg_daily_revenue"] = per_customer(win, "total_revenue") / features["tenure_days"]
# column to be dropped before modelling because it doesn't add any new information
features["avg_daily_revenue_30d"] = features["revenue_30d"] / 30
features["recent_vs_usual_revenue"] = features["avg_daily_revenue_30d"] / (features["avg_daily_revenue"] + 1)

# percentage change between equal-length periods
features["revenue_change_30d"] = pct_change(features["revenue_30d"],
                                            per_customer(window(60,30), "total_revenue"))
features["revenue_change_90d"] = pct_change(features["revenue_90d"],
                                            per_customer(window(180,90), "total_revenue"))

display(features.head())

In [ ]:
""" recharge features """

# number of recharge transactions: total and last 30 days
features["recharge_count_total"] = per_customer(win, "recharge_count")
features["recharge_count_30d"] = per_customer(window(30), "recharge_count")

# average recharge amount (0 for customers who never recharged)
recharge_total =  per_customer(win, "recharge_amount")
features["avg_recharge_amount"] = (recharge_total / features["recharge_count_total"].replace(0, np.nan)).fillna(0)

# typical days between recharges
recharge_days = win.loc[win["recharge_amount"] > 0, ["customer_id", "date"]].copy()
recharge_days["gap"] = recharge_days.groupby("customer_id", observed=True)["date"].diff().dt.days

features["median_recharge_gap"] = recharge_days.groupby("customer_id", observed=True)["gap"].median()

display(features.head())

In [ ]:
""" activity and recency """

# number of active days (days with revenue)
features["active_days_total"] = per_customer(win, "active")
features["active_days_30d"] = per_customer(window(30), "active")
features["active_day_share"] = features["active_days_total"] / features["tenure_days"]

# days since last qualifying activity (revenue) and since last recharge
last_revenue = win[win["active"]].groupby("customer_id", observed=True)["date"].max()
features["days_since_last_revenue"] = (cutoff - last_revenue).dt.days

last_recharge = win[win["recharge_amount"] > 0].groupby("customer_id", observed=True)["date"].max()
features["days_since_last_recharge"] = (cutoff - last_recharge).dt.days

display(features.head())

In [ ]:
""" revenue shares """

total_rev = per_customer(win, "total_revenue")
for col in ["data_revenue", "voice_revenue", "sms_revenue"]:
    features[col.replace("revenue", "revenue_share")]= per_customer(win, col) / total_rev

display(features.head())

In [ ]:
""" usage and purchase frequency trends """

# last 30 days vs the days before
prev = window(60, 30)
last = window(30)

features["active_days_change"] = pct_change(features["active_days_30d"], per_customer(prev, "active"))
features["recharge_count_change"] = pct_change(features["recharge_count_30d"], per_customer(prev, "recharge_count"))

for col in ["data_traffic", "voice_traffic"]:
    features[f"{col}_change"] = pct_change(per_customer(last, col), per_customer(prev, col))

# yes/no usage flags
features["uses_data"] = (per_customer(win, "data_traffic") > 0).astype(int)
features["pays_sms"] = (per_customer(win, "sms_revenue") > 0).astype(int)

display(features.head())

In [ ]:
""" fill gaps, attach label, check """

# customers with no recharge in the window: fill with their tenure
# (no recharge since the start of the observation window)
features["median_recharge_gap"] = features["median_recharge_gap"].fillna(features["tenure_days"])
features["days_since_last_recharge"] = features["days_since_last_recharge"].fillna(features["tenure_days"])

features = features.drop(columns="churned", errors="ignore").join(churn)

print(features.shape)
print("\n Missing values:")
print(features.isna().sum()[features.isna().sum() > 0])
display(features.describe().T.round(2))

### Churn Prediction Model

Goal: predict which customers are likely to churn in the next 90 days.

Churn definition: a customer has churned if they generate no revenue for 90 consecutive days.

Approach:
- The data is split at a cutoff date (2026-07-02):
  - Observation window (2025-09-01 to 2026-07-02): customer behavior, used to build features
  - Outcome window (2026-07-03 to 2026-09-30): used to label each customer as churned or not
- Only customers who generated revenue in the 90 days before the cutoff are included (1,924 customers); those already inactive are excluded.
- Churn rate: 16.5% (317 churned, 1,607 stayed).

Models:
- Logistic Regression: a simple, interpretable baseline
- Random Forest Classifier: a stronger model that captures non-linear patterns

Evaluation: precision, recall, F1-score, ROC-AUC and confusion matrix on an untouched test set. ROC-AUC is used to compare models because it measures ranking quality regardless of the decision threshold. The classification report and confusion matrix use the default threshold of 0.5; in practice the threshold should depend on the cost of a missed churner compared with a false alarm.


In [ ]:
"""
Comparing the observation-window behavior of customers who churned with those who stayed.
"""
compare_cols = ["avg_daily_revenue", "revenue_30d", "revenue_change_30d", "revenue_change_90d",
                "recharge_count_30d", "avg_recharge_amount", "median_recharge_gap",
                "active_days_30d", "active_day_share", "days_since_last_revenue",
                "days_since_last_recharge", "active_days_change", "recharge_count_change",
                "data_revenue_share", "voice_revenue_share", "tenure_days"]

comparison = features.groupby("churned")[compare_cols].median().T
comparison.columns = ["stayed (0)", "churned (1)"]
display(comparison.round(2))

flag_share = features.groupby("churned")[["uses_data", "pays_sms"]].mean()
display(flag_share.style.format("{:.1%}"))


In [ ]:
""" box plots """
plot_cols = ["avg_daily_revenue", "revenue_change_30d", "active_day_share",
             "days_since_last_revenue", "days_since_last_recharge", "recharge_count_change"]

n_rows = math.ceil(len(plot_cols) / 3)
fig, axes = plt.subplots(n_rows, 3, figsize=(16, n_rows * 4))
for ax, col in zip(axes.flat, plot_cols):
    sns.boxplot(data=features, x="churned", y=col, ax=ax, showfliers=False)
    ax.set_title(col)
for ax in axes.flat[len(plot_cols):]:
    ax.set_visible(False)

plt.suptitle("Observation-Window Behavior: Stayed (0) vs Churned (1)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
""" churn rate by group """

# churn rate by how long since the customer last paid
features["recency_bucket"] = pd.cut(features["days_since_last_revenue"],
                                    bins=[-1, 7, 14, 30, 60, 90],
                                    labels=["0-7", "8-14", "15-30", "31-60", "61-90"])
display(features.groupby("recency_bucket", observed=True)["churned"].agg(["mean", "size"]))

# churn rate by activity level (4 equal-sized groups)
features["activity_bucket"] = pd.qcut(features["active_day_share"], 4,
                                      labels=["low", "mid-low", "mid-high", "high"])
display(features.groupby("activity_bucket", observed=True)["churned"].agg(["mean", "size"]))

In [ ]:
""" churn correlation: which features relate most to churn """

churn_corr = (features[compare_cols + ["churned"]]
              .corr(method="spearman")["churned"]
              .drop("churned")
              .sort_values())
display(churn_corr.round(3))

In [ ]:
"""
Model Preparation
Separate features and target
"""

X = features.drop(columns=["churned", "recency_bucket", "activity_bucket", "avg_daily_revenue_30d"],
                  errors="ignore")
y = features["churned"]

# cap change features at 10: when the previous period was 0, any new activity
# produces extremely large values that would distort the model
change_cols = [c for c in X.columns if "change" in c]
X[change_cols] = X[change_cols].clip(upper=10)

print(X.shape, y.shape)

In [ ]:
""" train / validation / test split (60, 20, 20);
train and validation are combined later for cross-validation """


# first set aside the test set: untouched until evaluation
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# then split the rest into training and validation (0.25 of 80% = 20% of all)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)
print(f"Churn rate: train {y_train.mean():.1%} | validation {y_val.mean():.1%} | test {y_test.mean():.1%}")

In [ ]:
""" correlation between model features """

feature_corr = X_train.corr(method="spearman")

# hide the upper triangle since it mirrors the lower one
mask = np.triu(np.ones_like(feature_corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(feature_corr,
            mask=mask,
            annot=True,
            fmt=".1f",
            annot_kws={"size": 7},
            cmap="coolwarm",
            vmin=-1, vmax=1, center=0,
            linewidth=0.5,
            ax=ax)

ax.set_title("Spearman Correlation between Model Features (training set)")
plt.tight_layout()
plt.show()

# list the most strongly related pairs
pairs = (feature_corr.where(~mask)
         .stack()
         .rename("correlation")
         .reset_index())

pairs.columns = ["feature_1", "feature_2", "correlation"]
strong_pairs = (pairs[pairs["correlation"].abs() >= 0.7]
                .sort_values("correlation", key=abs, ascending=False))
display(strong_pairs.round(2))

In [ ]:
""" remove one feature from each very highly correlated pair """

drop_threshold = 0.9

# how strongly each feature relates to churn, on the training set only
train_churn_corr = X_train.corrwith(y_train, method="spearman").abs()

to_drop = set()
for _, row in strong_pairs[strong_pairs["correlation"].abs() >= drop_threshold].iterrows():
    f1, f2 =  row["feature_1"], row["feature_2"]
    if f1 in to_drop or f2 in to_drop:
        continue
    weaker = f1 if train_churn_corr[f1] < train_churn_corr[f2] else f2
    to_drop.add(weaker)

print("Dropped:", sorted(to_drop))
X_train = X_train.drop(columns=list(to_drop))
X_val =  X_val.drop(columns=list(to_drop))
X_test = X_test.drop(columns=list(to_drop))
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
""" log-transform skewed columns (chosen using the training set only) """

skewed_cols = [c for c in X_train.columns
               if X_train[c].min() >= 0 and X_train[c].nunique() > 2 and X_train[c].skew() > 1]

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()
X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_val[skewed_cols] = np.log1p(X_val[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

print("Log-transformation:", skewed_cols)


In [ ]:
""" skewness before and after the log transformation """

skew_check = pd.DataFrame({
    "before": features.loc[X_train.index, skewed_cols].skew(),
    "after": X_train[skewed_cols].skew(),
}).round(2).sort_values("after", ascending=False)

display(skew_check)


In [ ]:
""" logistic regression (with scaling): tune settings with 5-fold cross validation """

# training and validation together; cross-validation creates the validation folds
X_trval = pd.concat([X_train, X_val])
y_trval = pd.concat([y_train, y_val])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_results = []
for C in [0.001, 0.003, 0.01, 0.03, 0.1, 1, 10]:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=1000, class_weight="balanced")
    )

    scores = cross_val_score(model, X_trval, y_trval, cv=cv, scoring="roc_auc")
    lr_results.append({"C": C, "cv_auc": scores.mean(), "cv_std": scores.std()})

lr_results = pd.DataFrame(lr_results).sort_values("cv_auc", ascending=False)
display(lr_results.round(3))

# final logistic regression with the best C
best_C = lr_results.iloc[0]["C"]
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=best_C, max_iter=1000, class_weight="balanced")
)
log_reg.fit(X_trval, y_trval)
print("Best C:", best_C)

In [ ]:
""" Random Forest Classifier: tune settings with 5-fold cross-validation """

rf_grid = ParameterGrid({
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt",  0.5]
})

rf_results = []
best_auc, best_params = 0, None
for params in rf_grid:
    model = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=42, n_jobs=-1, **params)
    scores = cross_val_score(model, X_trval, y_trval, cv=cv, scoring="roc_auc")
    rf_results.append({**params, "cv_auc": scores.mean(), "cv_std": scores.std()})
    if scores.mean() > best_auc:
        best_auc, best_params = scores.mean(), params

display(pd.DataFrame(rf_results).sort_values("cv_auc", ascending=False).head(10).round(3))

# final random forest with the best settings
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=42, n_jobs=-1, **best_params)

rf.fit(X_trval, y_trval)
print("Best settings:", best_params)

In [ ]:
""" does predicted churn probability rise with days away? """

check = pd.DataFrame({
    "days_away": features.loc[X_test.index, "days_since_last_revenue"],
    "churn_probability": rf.predict_proba(X_test)[:, 1],
    "actually_churned": y_test,
})
check["days_away_group"] = pd.cut(check["days_away"], [-1, 7, 14, 30, 60, 90],
                                  labels=["0-7", "8-14", "15-30", "31-60", "61-90"])

display(check.groupby("days_away_group", observed=True)
        .agg(avg_churn_probability=("churn_probability", "mean"),
             actual_churn_rate=("actually_churned", "mean"),
             customers=("days_away", "size"))
        .round(3))


In [ ]:
""" compare the tuned models (cross-validation) """

print(f"{'Logistic Regression':<20} CV ROC-AUC: {lr_results['cv_auc'].max():.3f}")
print(f"{'Random Forest':<20} CV ROC-AUC: {best_auc:.3f}")


In [ ]:
""" Evaluate both models """

for name, model in [("Logistic Regression", log_reg), ("Random Forest", rf)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"=== {name} === \n")
    print(classification_report(y_test, y_pred, target_names=["stayed", "churned"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}\n")

    with sns.axes_style("white"):
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred,
            display_labels=["stayed", "churned"],
            cmap="Blues",
            ax=ax)
        plt.title(name)
        plt.show()


In [ ]:
""" churn probability for each eligible subscriber """

# customers used for training get out-of-fold probabilities:
# each one is scored by a model that did not see them during training
rf_template = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                     random_state=42, n_jobs=-1, **best_params)
oof_prob = cross_val_predict(rf_template, X_trval, y_trval, cv=cv, method="predict_proba")[:, 1]

churn_scores = pd.concat([
    pd.DataFrame({"churn_probability": oof_prob, "scored_on": "cross-validation"},
                 index=X_trval.index),
    pd.DataFrame({"churn_probability": rf.predict_proba(X_test)[:, 1], "scored_on": "test"},
                 index=X_test.index),
])
churn_scores["churned"] = y.loc[churn_scores.index]
# provisional bands on raw scores; replaced by calibrated bands in the next cells
churn_scores["risk_band"] = pd.cut(churn_scores["churn_probability"], bins=[0, 0.3, 0.6, 1],
                                   labels=["low", "medium", "high"], include_lowest=True)
churn_scores = churn_scores.sort_values("churn_probability", ascending=False)

print(churn_scores.shape)
display(churn_scores.head(10))
display(churn_scores.groupby("risk_band", observed=True)["churned"]
        .agg(customers="size", actual_churn_rate="mean").round(3))



In [ ]:
""" Calibrating the churn probabilities

The random forest scores rank customers well but overstate the chance of churning
(average score 33.6% against an actual churn rate of 16.5%),
because class weighting makes the model treat churners as half of all customers.

Calibration fits a sigmoid curve that maps the scores to observed churn rates,
using customers the forest did not train on.
It changes the probability values but keeps the order of customers almost unchanged.

Checks:
- Average probability compared with the actual churn rate
- Brier score (average squared gap between probability and outcome, lower is better)
- Test ROC-AUC before and after
- Reliability table: average probability against actual churn rate within score groups

"""

# random forest with sigmoid calibration, fitted with 5 inner folds
cal_template = CalibratedClassifierCV(estimator=rf_template, method="sigmoid", cv=5)

# same scheme as churn_scores: out-of-fold for train+val, a final model for the test set
oof_cal = cross_val_predict(cal_template, X_trval, y_trval, cv=cv, method="predict_proba")[:, 1]
rf_cal = CalibratedClassifierCV(estimator=rf_template, method="sigmoid", cv=5).fit(X_trval, y_trval)
test_cal = rf_cal.predict_proba(X_test)[:, 1]

compare = pd.DataFrame({
    "raw": pd.concat([pd.Series(oof_prob, index=X_trval.index),
                      pd.Series(rf.predict_proba(X_test)[:, 1], index=X_test.index)]),
    "calibrated": pd.concat([pd.Series(oof_cal, index=X_trval.index),
                             pd.Series(test_cal, index=X_test.index)]),
})
compare["churned"] = y.loc[compare.index]

# 1) does the average score match the real churn rate?
print(f"Actual churn rate: {compare['churned'].mean():.1%}")
print(f"Average score: raw {compare['raw'].mean():.1%} | calibrated {compare['calibrated'].mean():.1%}")

# 2) Brier score: average squared gap between score and outcome (lower = better)
print(f"Brier: raw {brier_score_loss(compare['churned'], compare['raw']):.3f} | "
      f"calibrated {brier_score_loss(compare['churned'], compare['calibrated']):.3f}")

# 3) ranking should barely change
print(f"Test AUC: raw {roc_auc_score(y_test, compare.loc[X_test.index, 'raw']):.3f} | "
      f"calibrated {roc_auc_score(y_test, test_cal):.3f}")

# 4) reliability table: in each score group, average score vs real churn rate
for col in ["raw", "calibrated"]:
    groups = pd.cut(compare[col], bins=[0, 0.1, 0.2, 0.3, 0.45, 0.6, 1], include_lowest=True)
    print(col)
    display(compare.groupby(groups, observed=True)
            .agg(avg_score=(col, "mean"), actual_churn_rate=("churned", "mean"),
                 customers=("churned", "size")).round(3))


In [ ]:
""" calibrated churn probabilities and risk categories """

churn_scores["raw_score"] = compare["raw"]
churn_scores["churn_probability"] = compare["calibrated"]

# risk categories on calibrated probabilities: low < 10%, medium 10-30%, high 30%+
churn_scores["risk_band"] = pd.cut(churn_scores["churn_probability"], bins=[0, 0.1, 0.3, 1],
                                   labels=["low", "medium", "high"], include_lowest=True)
churn_scores = churn_scores.sort_values("churn_probability", ascending=False)

display(churn_scores.groupby("risk_band", observed=True)
        .agg(customers=("churned", "size"),
             avg_probability=("churn_probability", "mean"),
             actual_churn_rate=("churned", "mean")).round(3))


In [ ]:
""" gains chart: share of churners caught when contacting the riskiest customers first """

ranked = churn_scores.sort_values("churn_probability", ascending=False)
contacted = np.arange(1, len(ranked) + 1) / len(ranked)        # share of customers contacted so far
caught = ranked["churned"].cumsum() / ranked["churned"].sum()  # share of all churners caught so far

# churners caught when contacting the riskiest 10%, 20%, 30% and 50%
gains_table = pd.DataFrame({
    "customers_contacted": [int(round(len(ranked) * p)) for p in [0.1, 0.2, 0.3, 0.5]],
    "share_contacted": [0.1, 0.2, 0.3, 0.5],
})
gains_table["share_of_churners_caught"] = [caught.iloc[n - 1] for n in gains_table["customers_contacted"]]
gains_table["lift"] = gains_table["share_of_churners_caught"] / gains_table["share_contacted"]
display(gains_table.round(2))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(contacted, caught.values, label="Random forest (calibrated)")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random contact")
ax.set_xlabel("Share of customers contacted (riskiest first)")
ax.set_ylabel("Share of all churners caught")
ax.set_title("Gains Chart: Churn Model")
ax.legend()
plt.tight_layout()
plt.show()


### Customer Lifetime Value (CLV) Forecasting

Goal: estimate the revenue each customer is expected to generate over the next 90 days.

CLV definition: predicted future revenue over the forecast period, not customer profit (costs are not included).

Approach:
- The data is split at a cutoff date (2026-04-03), 90 days before the churn cutoff:
  - Feature window (2025-10-06 to 2026-04-03): the 180 days before the cutoff, used to build features
  - Target window (2026-04-04 to 2026-07-02): 90 days, used to measure each customer's actual future revenue
- The 90-day horizon matches the churn outcome window, so both models describe the same period.
- The target window ends at the churn cutoff, so predictions made at the churn cutoff can be checked against what customers actually spent afterwards (2026-07-03 to 2026-09-30).
- Features use a fixed 180-day lookback so they keep the same meaning when the model is applied at later dates.
- Only customers who generated revenue in the 90 days before the cutoff are included (2,053 customers), the same rule as for churn.
- 20.4% of these customers generated no revenue in the target window.

Models:
- Naive baseline: each customer's revenue in the last 90 days, used unchanged as the forecast. It assumes customers keep spending at their recent pace. Windows of 30, 60 and 90 days gave similar errors on the training data (within 2%); 90 days was kept because it matches the forecast horizon.
- Random Forest Regressor: captures non-linear patterns and interactions between features

Evaluation:
- MAE, RMSE and R² on a held-out test set, compared against the naive baseline
- Ranking quality: Spearman correlation and the share of the real top 20% each model finds
- Later-period check: predictions made at the churn cutoff compared with actual revenue in the following 90 days

In [ ]:
""" build features and targets for any cutoff date """

def eligible_at(cutoff):
    """ customers with revenue in the 90 days before the cutoff """
    recent = df[(df["date"] <= cutoff) & (df["date"] > cutoff - pd.Timedelta(days=90))]
    active =  recent.groupby("customer_id", observed=True)["total_revenue"].sum() > 0
    return active[active].index

def future_revenue(cutoff, days, customers):
    """ total revenue per customer in the 'days' after the cutoff """
    future = df[(df["date"] > cutoff) & (df["date"] <= cutoff + pd.Timedelta(days=days))]
    return (future.groupby("customer_id", observed=True)["total_revenue"].sum()
            .reindex(customers, fill_value=0))

def build_features(cutoff, customers, lookback=180):
    """
    one row per customer, using only data from the 'lookback' days up to the 'cutoff'
    """
    win = df[(df["customer_id"].isin(customers)) &
             (df["date"] <= cutoff) &
             (df["date"] > cutoff - pd.Timedelta(days=lookback))]
    feats = pd.DataFrame(index=customers)

    def window(days_from, days_to=0):
        return win[(win["date"] > cutoff - pd.Timedelta(days=days_from)) &
                   (win["date"] <= cutoff - pd.Timedelta(days=days_to))]
    
    def per_customer(data, col, how="sum"):
        return (data.groupby("customer_id", observed=True)[col]
                .agg(how)
                .reindex(customers, fill_value=0))

    # revenue
    first_seen = win.groupby("customer_id", observed=True)["date"].min()
    feats["tenure_days"] = (cutoff - first_seen).dt.days + 1
    for d in [30, 60, 90]:
        feats[f"revenue_{d}d"] = per_customer(window(d), "total_revenue")
    feats["avg_daily_revenue"] = per_customer(win, "total_revenue") / feats["tenure_days"]
    feats["recent_vs_usual_revenue"] = (feats["revenue_30d"] / 30) / (feats["avg_daily_revenue"] + 1)
    feats["revenue_change_30d"] = pct_change(feats["revenue_30d"],
                                             per_customer(window(60, 30), "total_revenue"))
    feats["revenue_change_90d"] = pct_change(feats["revenue_90d"],
                                              per_customer(window(180, 90), "total_revenue"))

    # recharge
    feats["recharge_count_total"] = per_customer(win, "recharge_count")
    feats["recharge_count_30d"] = per_customer(window(30), "recharge_count")
    recharge_total = per_customer(win, "recharge_amount")
    feats["avg_recharge_amount"] = (recharge_total / feats["recharge_count_total"].replace(0, np.nan)).fillna(0)
    recharge_days = win.loc[win["recharge_amount"] > 0, ["customer_id", "date"]].copy()
    recharge_days["gap"] = recharge_days.groupby("customer_id", observed=True)["date"].diff().dt.days
    feats["median_recharge_gap"] = recharge_days.groupby("customer_id", observed=True)["gap"].median()

    # activity and recency
    feats["active_days_total"] = per_customer(win, "active")
    feats["active_days_30d"] = per_customer(window(30), "active")
    feats["active_day_share"] = feats["active_days_total"] / feats["tenure_days"]
    last_revenue = win[win["active"]].groupby("customer_id", observed=True)["date"].max()
    feats["days_since_last_revenue"] = (cutoff - last_revenue).dt.days
    last_recharge = win[win["recharge_amount"] > 0].groupby("customer_id", observed=True)["date"].max()
    feats["days_since_last_recharge"] = (cutoff - last_recharge).dt.days

    # revenue shares
    total_rev = per_customer(win, "total_revenue")
    for col in ["data_revenue", "voice_revenue", "sms_revenue"]:
        feats[col.replace("revenue", "revenue_share")] = per_customer(win, col) / total_rev

    # trends: last 30 days vs the 30 days before
    prev, last = window(60, 30), window(30)
    feats["active_days_change"] = pct_change(feats["active_days_30d"], per_customer(prev, "active"))
    feats["recharge_count_change"] = pct_change(feats["recharge_count_30d"], 
                                                per_customer(prev, "recharge_count"))
    for col in ["data_traffic", "voice_traffic"]:
        feats[f"{col}_change"] = pct_change(per_customer(last, col), per_customer(prev, col))

    # yes/no flags
    feats["uses_data"] = (per_customer(win, "data_traffic") > 0).astype(int)
    feats["pays_sms"] = (per_customer(win, "sms_revenue") > 0).astype(int)

    # no recharge in the lookback: fill with tenure
    feats["median_recharge_gap"] = feats["median_recharge_gap"].fillna(feats["tenure_days"])
    feats["days_since_last_recharge"] = feats["days_since_last_recharge"].fillna(feats["tenure_days"])

    return feats

In [ ]:
""" CLV dataset: features at the cutoff, revenue over the next 90 days as target """

horizon = 90  # same length as the churn outcome window
clv_cutoff = cutoff - pd.Timedelta(days=horizon)  # 3 Apr 2026: the target window ends at the scoring date

clv_customers = eligible_at(clv_cutoff)
clv_features = build_features(clv_cutoff, clv_customers)
clv_target = future_revenue(clv_cutoff, horizon, clv_customers).rename("future_revenue")

print(f"Feature window: {(clv_cutoff - pd.Timedelta(days=179)).date()} to {clv_cutoff.date()}")
print(f"Target window: {(clv_cutoff + pd.Timedelta(days=1)).date()} to "
      f"{(clv_cutoff + pd.Timedelta(days=horizon)).date()}")
print(f"Total customers eligible at cutoff: {len(clv_customers)}")
print(f"Customers with zero future revenue: {(clv_target == 0).mean():.1%}")
display(clv_target.describe().round(2))


In [ ]:
""" check: build_features gives the same values as the churn feature cells """

check_feats = build_features(cutoff, eligible, lookback=1000)
diff = (check_feats - features[check_feats.columns]).abs().max()
print("Columns that differ:", list(diff[diff > 1e-9].index))


In [ ]:
""" CLV model preparation and train/test split """

Xc= clv_features.copy()
yc = clv_target

# cap change features at 10
change_cols_c = [c for c in Xc.columns if "change" in c]
Xc[change_cols_c] = Xc[change_cols_c].clip(upper=10)

# stratify on zero / non-zero future revenue
Xc_train, Xc_test, yc_train, yc_test =  train_test_split(
    Xc, yc, test_size=0.2, stratify=(yc > 0), random_state=42
)

print("Train:", Xc_train.shape, "| Test:", Xc_test.shape)
print(f"Zero future revenue: train {(yc_train == 0).mean():.1%} | test {(yc_test == 0).mean():.1%}")
print(f"Mean future revenue: train {yc_train.mean():,.0f} | test {yc_test.mean():,.0f}")

In [ ]:
""" random forest regressor: tune settings with 5-fold cross-validation """

cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

rf_reg_grid = ParameterGrid({
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5],
})

rf_reg_results = []
best_mae, best_reg_params = np.inf, None
for params in rf_reg_grid:
    model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, **params)
    mae = -cross_val_score(model, Xc_train, yc_train, cv=cv_reg,
                           scoring="neg_mean_absolute_error").mean()
    rf_reg_results.append({**params, "cv_mae": mae})
    if mae < best_mae:
        best_mae, best_reg_params = mae, params

display(pd.DataFrame(rf_reg_results).sort_values("cv_mae").head(10).round(0))

rf_reg = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, **best_reg_params)
rf_reg.fit(Xc_train, yc_train)
print("Best settings:", best_reg_params)

In [ ]:
""" which history window makes the best naive baseline? (training data) """

train_raw = clv_features.loc[Xc_train.index] # original values, before clipping

windows = {
    "last 30 days": train_raw["revenue_30d"] * horizon / 30,
    "last 60 days": train_raw["revenue_60d"] * horizon / 60,
    "last 90 days": train_raw["revenue_90d"] * horizon / 90,
    "last 180 days (daily average)": train_raw["avg_daily_revenue"] * horizon,
}
for name, pred in windows.items():
    print(f"{name:<32} MAE: {mean_absolute_error(yc_train, pred):,.0f}")

In [ ]:
""" compare: naive baseline vs random forest, on the training data """

baseline_mae = mean_absolute_error(yc_train, train_raw["revenue_90d"] * horizon / 90)

print(f"{'Naive baseline':<20} MAE: {baseline_mae:,.0f}")
print(f"{'Random Forest (CV)':<20} MAE: {best_mae:,.0f}")

In [ ]:
""" final evaluation on the test set """

def regression_scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

test_raw = clv_features.loc[Xc_test.index]
predictions = {
    "Naive baseline": test_raw["revenue_90d"] * horizon / 90,
    "Random Forest": rf_reg.predict(Xc_test),
}

clv_results = pd.DataFrame({name: regression_scores(yc_test, pred)
                            for name, pred in predictions.items()}).T
display(clv_results.round(2))

In [ ]:
""" do the models underestimate top spenders, and do they rank customers well? """

check_clv = pd.DataFrame({
    "actual": yc_test,
    "baseline": test_raw["revenue_90d"] * horizon / 90,
    "random_forest": rf_reg.predict(Xc_test),
}, index=yc_test.index)

# the 10 highest actual spenders
display(check_clv.sort_values("actual", ascending=False).head(10).round(0))

# how well each model ranks customers (1 = perfect order)
print(check_clv.corr(method="spearman")["actual"].round(3))


In [ ]:
""" how many of the real top 20% does each model put in its top 20%? """

top_n = int(len(check_clv) * 0.2)
actual_top = set(check_clv["actual"].nlargest(top_n).index)

for model in ["baseline", "random_forest"]:
    predicted_top = set(check_clv[model].nlargest(top_n).index)
    overlap = len(actual_top & predicted_top) / top_n
    print(f"{model:<15} finds {overlap:.0%} of the real top 20%")

### Retention Prioritization

Goal: give each eligible subscriber a churn probability, predicted future revenue, a risk category, a value category and a proposed retention priority, by combining the outputs of the churn and CLV models.

Approach:
- Churn risk: calibrated churn probability from the churn model, at the churn cutoff (2026-07-02).
- Risk category: low (under 10%), medium (10% to 30%) and high (30% or more) calibrated churn probability. The average churn rate is 16.5%, so low is clearly below average and high is about double the average.
- Customer value: predicted revenue over the next 90 days from the CLV model, using features built at the same cutoff with the same 180-day lookback.
- Value category: measured two ways, each split into bottom 50% = low, next 30% = medium, top 20% = high: predicted revenue over the next 90 days, and revenue 61 to 180 days before the cutoff. A customer's value category is the higher of the two. Spending drops in the 60 days before churn, so predicted revenue alone would rank customers who were valuable until recently as low value (see the finding below).
- Medium value is included with high value because only 29 high-risk customers are high value, medium value customers are still in the top half of spenders, and some only look medium because their spending is already falling.
- Expected revenue at risk: each customer's calibrated churn probability × predicted CLV. It is used to rank customers within a priority group, not as a revenue forecast, because predicted CLV already reflects part of the churn risk.
- Segments (description): high or low risk × high or low value, where high risk = high risk category and high value = medium or high value category.
- Retention priority (action):
  - Retain now: high risk, medium or high value
  - Monitor closely: medium risk, medium or high value
  - Low-cost action: high or medium risk, low value
  - Maintain: low risk

Output: one row per eligible subscriber, ranked by priority and then by expected revenue at risk, plus summaries per priority and per segment.


In [ ]:
""" predicted CLV at the churn cutoff, for the customers in the churn analysis """

# same 180-day lookback and preparation as the CLV training data
ret_features = build_features(cutoff, eligible)
ret_features[change_cols_c] = ret_features[change_cols_c].clip(upper=10)

predicted_clv = pd.Series(rf_reg.predict(ret_features[Xc_train.columns]),
                          index=ret_features.index, name="predicted_clv")
print(f"Customers: {len(predicted_clv)}")
display(predicted_clv.describe().round(0))

In [ ]:
""" out-of-time check: predictions made at 2 Jul vs what customers actually spent from 3 Jul to 30 Sep """

actual_next_90d = future_revenue(cutoff, horizon, eligible)
oot = pd.DataFrame({
    "actual": actual_next_90d,
    "baseline": ret_features["revenue_90d"],  # last 90 days, same length as the horizon
    "random_forest": predicted_clv,
})

display(pd.DataFrame({name: regression_scores(oot["actual"], oot[name])
                      for name in ["baseline", "random_forest"]}).T.round(2))
print(oot.corr(method="spearman")["actual"].round(3))


In [ ]:
""" combine churn risk and predicted value; expected revenue at risk """

retention = churn_scores.join(predicted_clv)
retention["past_6m_revenue"] = ret_features["avg_daily_revenue"] * ret_features["tenure_days"]
print("Missing CLV values:", retention["predicted_clv"].isna().sum())

# expected revenue at risk: each customer's calibrated probability × predicted CLV
retention["revenue_at_risk"] = retention["churn_probability"] * retention["predicted_clv"]

# actual churn rate of each risk category, used in the summaries and the app
band_rate = retention.groupby("risk_band", observed=True)["churned"].mean()
retention["band_churn_rate"] = retention["risk_band"].map(band_rate).astype(float)

# check: calibrated probabilities should add up to roughly the real number of churners
print(f"Expected churners: {retention['churn_probability'].sum():.0f} | actual: {retention['churned'].sum()}")

display(band_rate.round(3))
display(retention.head())


In [ ]:
""" risk category, value category, and retention priority """

retention = retention.rename(columns={"risk_band": "risk_category"})

# value tiers: bottom 50% low, next 30% medium, top 20% high, measured two ways
value_levels = ["low", "medium", "high"]

# predicted revenue over the next 90 days
retention["predicted_value_category"] = pd.qcut(retention["predicted_clv"], q=[0, 0.5, 0.8, 1],
                                                labels=value_levels)

# revenue 61-180 days before the cutoff, before the drop seen in the last 60 days
retention["revenue_before_drop"] = retention["past_6m_revenue"] - ret_features["revenue_60d"]
retention["past_value_category"] = pd.qcut(retention["revenue_before_drop"].rank(method="first"),
                                           q=[0, 0.5, 0.8, 1], labels=value_levels)

# overall value category: the higher of the two
codes = np.maximum(retention["predicted_value_category"].cat.codes,
                   retention["past_value_category"].cat.codes)
retention["value_category"] = pd.Categorical.from_codes(codes.to_numpy(), categories=value_levels,
                                                        ordered=True)


high_risk = retention["risk_category"] == "high"
medium_risk = retention["risk_category"] == "medium"
# "high_value" here means medium or high value (Retain now includes medium value)
high_value = retention["value_category"].isin(["medium", "high"])

segment_order = ["High risk, high value", "Low risk, high value",
                "High risk, low value", "Low risk, low value" ]
retention["segment"] = np.select(
    [high_risk & high_value, ~high_risk & high_value, high_risk & ~high_value],
    segment_order[:3],
    default=segment_order[3],
)
retention["segment"] = pd.Categorical(retention["segment"], categories=segment_order, ordered=True)

# retention priority: segments plus extra attention for medium-risk customers
priority_order = ["Retain now", "Monitor closely", "Low-cost action", "Maintain"]
retention["retention_priority"] = np.select(
    [high_risk & high_value, medium_risk & high_value, (high_risk | medium_risk) & ~high_value],
    priority_order[:3],
    default=priority_order[3],
)
retention["retention_priority"] = pd.Categorical(retention["retention_priority"], 
                                                 categories=priority_order, ordered=True)

# how risk and value overlap
display(pd.crosstab(retention["risk_category"], retention["value_category"]))

# how segments and priorities overlap
display(pd.crosstab(retention["segment"], retention["retention_priority"]))

display(pd.crosstab(retention["predicted_value_category"], retention["past_value_category"]))


In [ ]:
""" check: how many high-risk customers were valuable by their past revenue? """
past_value = pd.qcut(retention["past_6m_revenue"], q=[0, 0.5, 0.8, 1], labels=["low", "medium", "high"])
display(pd.crosstab(retention["risk_category"], past_value, margins=True))

In [ ]:
""" when does spending start to drop? revenue per 30-day block before the churn cutoff """

blocks = {}
for k in range(5, -1, -1):  # oldest block first
    start_day, end_day = 30 * k + 1, 30 * (k + 1)
    part = df[(df["date"] > cutoff - pd.Timedelta(days=end_day)) &
              (df["date"] <= cutoff - pd.Timedelta(days=start_day - 1)) &
              (df["customer_id"].isin(eligible))]
    blocks[f"days {start_day}-{end_day}"] = (part.groupby("customer_id", observed=True)["total_revenue"]
                                             .sum().reindex(eligible, fill_value=0))
blocks = pd.DataFrame(blocks)

# average revenue per block, by risk category and by what actually happened
display(blocks.groupby(retention["risk_category"], observed=True).mean().round(0))
display(blocks.groupby(retention["churned"]).mean().round(0))

# how many high-risk customers count as medium/high value under each past window
candidates = {
    "last 90 days": blocks.iloc[:, -3:].sum(axis=1),
    "last 180 days": blocks.sum(axis=1),
    "days 61-180 (skips last 60)": blocks.iloc[:, :4].sum(axis=1),
    "days 91-180 (skips last 90)": blocks.iloc[:, :3].sum(axis=1),
}
for name, revenue in candidates.items():
    value = pd.qcut(revenue.rank(method="first"), q=[0, 0.5, 0.8, 1], labels=["low", "medium", "high"])
    n = (value.isin(["medium", "high"]) & (retention["risk_category"] == "high")).sum()
    print(f"{name:<30} high-risk customers with medium/high value: {n}")


**Finding: predicted value hides high-risk customers who were valuable**

Predicted CLV is based mostly on recent spending. To check whether this understates high-risk customers, their past revenue was compared with their predicted value, and their spending was tracked in 30-day blocks before the churn cutoff.

Results:
- By predicted 90-day revenue, only 12 of the 405 high-risk customers are medium or high value. By revenue over the previous six months, 96 are.
- Customers who churned kept stable spending until about 60 days before the cutoff (about 660 to 800 per 30 days), then it fell sharply (587, then 367). High-risk customers fell from 789 to 16 per 30 days over the same period. Customers who stayed kept steady or rising spending.
- Measuring past value on revenue 61 to 180 days before the cutoff, which ends where the drop begins, gives 124 high-risk customers with medium or high value. Skipping the last 90 days gives almost the same result (125), while windows that include the drop give fewer (65 for the last 90 days, 96 for the last 180 days).
- Even before the drop, high-risk customers spent less on average than low-risk customers (about 700 against 2,200 to 3,000 per 30 days).

Conclusion: because spending falls in the 60 days before churn, value based only on predicted revenue misses most high-risk customers who were valuable. The value category used for retention priorities is therefore the higher of predicted value and value based on revenue 61 to 180 days before the cutoff. With this measure, "Retain now" has 127 customers, compared with 12 using predicted value alone.


In [ ]:
""" summary per retention priority and per segment """

def retention_summary(group_col):
    summary = (retention.groupby(group_col, observed=True)
                    .agg(customers=("predicted_clv", "size"),
                            avg_predicted_clv=("predicted_clv", "mean"),
                            total_predicted_clv=("predicted_clv", "sum"),
                            revenue_at_risk=("revenue_at_risk", "sum"),
                            avg_past_6m_revenue=("past_6m_revenue", "mean"),
                            actual_churn_rate=("churned", "mean")))

    summary["share_of_customers"] = summary["customers"] / summary["customers"].sum()
    summary["share_of_revenue_at_risk"] = (summary["revenue_at_risk"]
                                                / summary["revenue_at_risk"].sum())
    return summary.style.format({
        "avg_predicted_clv": "{:,.0f}", "total_predicted_clv": "{:,.0f}", "revenue_at_risk": "{:,.0f}",
        "avg_past_6m_revenue": "{:,.0f}", "actual_churn_rate": "{:.1%}",
        "share_of_customers": "{:.1%}", "share_of_revenue_at_risk": "{:.1%}",
    })



display(retention_summary("retention_priority"))
display(retention_summary("segment"))

In [ ]:
""" retention output: one row per eligible subscriber """

retention_output = (retention[["churn_probability", "predicted_clv", "risk_category",
                               "value_category", "segment", "retention_priority", "revenue_at_risk"]]
                    .sort_values(["retention_priority", "revenue_at_risk"], ascending=[True, False]))
retention_output["priority_rank"] = range(1, len(retention_output) + 1)

print(retention_output.shape)
display(retention_output.head(20).round(2))

In [ ]:
""" export results for the local Streamlit app"""

# customer-level retention output
export = retention[["churn_probability", "predicted_clv", "past_6m_revenue",
                    "risk_category", "value_category", "segment",
                    "retention_priority", "band_churn_rate", "revenue_at_risk", "churned",
                    "predicted_value_category", "past_value_category", "revenue_before_drop"]].copy()
export = export.sort_values(["retention_priority", "revenue_at_risk"], ascending=[True, False])
export["priority_rank"] = range(1, len(export) + 1)
export.index.name = "customer_id"
export.to_csv("retention_output.csv")

# churn test set: real outcome and the final model's score
pd.DataFrame({"churned": y_test,
              "logistic_regression": log_reg.predict_proba(X_test)[:, 1],
              "random_forest": rf.predict_proba(X_test)[:, 1]}).to_csv("churn_test.csv")


# CLV test set: actual revenue, baseline and random forest predictions
check_clv.to_csv("clv_test.csv")

# feature importance for both models (no customer data)
importance = pd.concat({
    "churn": pd.Series(rf.feature_importances_, index=rf.feature_names_in_),
    "clv": pd.Series(rf_reg.feature_importances_, index=rf_reg.feature_names_in_),
}, names=["model", "feature"]).rename("importance")
importance.to_csv("feature_importance.csv")

print(export.shape, "| churn test:", len(y_test), "| CLV test:", len(check_clv))


In [ ]:
""" export aggregated EDA insights for the Streamlit app (summary numbers only, no customer rows) """
import json

# headline numbers
overview = {
    "customers": int(n_customers),
    "start": str(start.date()),
    "end": str(end.date()),
    "days": int(n_days),
    "paying_customers": int(len(active_customers)),
    "cutoff": str(cutoff.date()),
    "churn_customers": int(len(churn)),
    "dormant_at_cutoff": int((~active_at_cutoff).sum()),
    "churn_rate": float(churn.mean()),
}
with open("eda_overview.json", "w") as f:
    json.dump(overview, f, indent=2)

# month by month
pd.DataFrame({
    "month": month_labels,
    "paying_customers": active_per_month.values,
    "total_revenue": monthly.sum().values,
    "median_revenue_per_payer": median_per_month.values,
}).to_csv("eda_monthly.csv", index=False)

# revenue mix (data, voice, sms)
(customer_df[revenue_cols].sum()
 .rename_axis("revenue_type").rename("revenue").reset_index()
 .to_csv("eda_revenue_mix.csv", index=False))

# customer trend groups (growing, declining, ...)
(trend.value_counts()
 .rename_axis("trend").rename("customers").reset_index()
 .to_csv("eda_trends.csv", index=False))

# share of inactive streaks that ended with the customer coming back
pd.DataFrame([{"days_inactive": t,
               "streaks": int((runs["length"] >= t).sum()),
               "came_back": runs.loc[runs["length"] >= t, "returned"].mean()}
              for t in [7, 14, 30, 60, 90, 120, 150]]).to_csv("eda_inactivity.csv", index=False)

# churn indicators
comparison.rename_axis("feature").to_csv("eda_churn_medians.csv")
(churn_corr.rename_axis("feature").rename("correlation").reset_index()
 .to_csv("eda_churn_corr.csv", index=False))
for col, file_name in [("recency_bucket", "eda_churn_by_recency.csv"),
                       ("activity_bucket", "eda_churn_by_activity.csv")]:
    (features.groupby(col, observed=True)["churned"]
     .agg(churn_rate="mean", customers="size")
     .rename_axis("group").reset_index()
     .to_csv(file_name, index=False))

print("EDA files exported")


In [ ]:
""" export extra EDA insights for the app's Customer behavior page (aggregated only, no customer rows) """

# single numbers for the KPI cards
revenue_total = customer_df[revenue_cols].sum(axis=1)
facts = {
    "rows": int(len(df)),
    "expected_rows": int(n_customers * n_days),
    "missing_dates": int(len(missing_dates)),
    "customers_full_history": int((rows_per_customer == n_days).sum()),
    "min_days_per_customer": int(rows_per_customer.min()),
    "zero_data_revenue": int(zero_data_revenue.sum()),
    "zero_data_revenue_no_traffic": int((zero_data_revenue & (customer_df["data_traffic"] == 0)).sum()),
    "data_without_data_revenue": int(bundle_users.sum()),
    "data_without_data_revenue_with_voice": int((bundle_users & (customer_df["voice_revenue"] > 0)).sum()),
    "data_without_any_revenue": int((bundle_users & (revenue_total == 0)).sum()),
    "total_recharge": float(customer_df["recharge_amount"].sum()),
    "total_revenue": float(revenue_total.sum()),
    "gap_positive": int((spend_gap > 0).sum()),
    "gap_zero": int((spend_gap == 0).sum()),
    "gap_negative": int((spend_gap < 0).sum()),
    "gap_zero_inactive": int(((spend_gap == 0) & (customer_df["recharge_amount"] == 0)).sum()),
    "gap_median": float(spend_gap.median()),
    "gap_median_bundle_users": float(spend_gap[bundle_users].median()),
    "gap_median_others": float(spend_gap[~bundle_users].median()),
    "outlier_customers": int(outliers.any(axis=1).sum()),
    "outliers_by_measure": {col: int(n) for col, n in outliers.sum().items()},
    "streak_count": int(len(runs)),
    "streak_median_days": float(runs["length"].median()),
    "recharge_gap_median": float(median_gap.median()),
    "recharge_gap_q1": float(median_gap.quantile(0.25)),
    "recharge_gap_q3": float(median_gap.quantile(0.75)),
    "dormant_paid_before": int(ever_paid.sum()),
    "dormant_paid_before_came_back": float(reactivated[ever_paid].mean()),
    "dormant_never_paid": int((~ever_paid).sum()),
    "dormant_never_paid_came_back": float(reactivated[~ever_paid].mean()),
}
with open("eda_facts.json", "w") as f:
    json.dump(facts, f, indent=2)

# daily totals across all customers, plus customers with revenue per day
daily_out = daily.copy()
daily_out["active_customers"] = df.groupby("date")["active"].sum()
daily_out.rename_axis("date").reset_index().to_csv("eda_daily.csv", index=False)

# distributions: summary table and histogram counts on a log scale (customers above zero)
dist_summary.rename_axis("measure").to_csv("eda_distribution_summary.csv")
totals = customer_df.assign(total_revenue=revenue_total)
hist_rows = []
for col in ["total_revenue", "data_revenue", "voice_revenue", "recharge_amount",
            "recharge_count", "data_traffic", "voice_traffic"]:
    values = totals.loc[totals[col] > 0, col]
    counts, edges = np.histogram(np.log10(values), bins=30)
    hist_rows += [{"measure": col, "from": 10 ** a, "to": 10 ** b, "customers": int(n)}
                  for a, b, n in zip(edges[:-1], edges[1:], counts)]
pd.DataFrame(hist_rows).to_csv("eda_histograms.csv", index=False)

# correlations between customer totals (customers with revenue)
(active_customers.corr(method="spearman").stack()
 .rename_axis(["measure_1", "measure_2"]).rename("correlation").reset_index()
 .to_csv("eda_correlations.csv", index=False))

# share of days with revenue, in 10% steps
active_share = df.groupby("customer_id", observed=True)["active"].mean()
share_edges = np.linspace(0, 1, 11)
share_labels = [f"{a:.0%}-{b:.0%}" for a, b in zip(share_edges[:-1], share_edges[1:])]
(pd.cut(active_share, bins=share_edges, labels=share_labels, include_lowest=True)
 .value_counts(sort=False).rename_axis("active_share").rename("customers").reset_index()
 .to_csv("eda_active_share.csv", index=False))

print("Extra EDA files exported:", len(facts), "facts")


In [ ]:
""" export customer-level activity for the app's customer lookup """

lookup_ids = retention.index

# monthly activity per customer (whole period; the model only saw the months before the scoring date)
sub = df[df["customer_id"].isin(lookup_ids)]
customer_activity = (sub.groupby(["customer_id", sub["date"].dt.to_period("M")], observed=True)
                     .agg(revenue=("total_revenue", "sum"),
                          active_days=("active", "sum"),
                          recharges=("recharge_count", "sum"),
                          recharge_amount=("recharge_amount", "sum"))
                     .reset_index())
customer_activity["month"] = customer_activity["date"].dt.strftime("%Y-%m")
customer_activity.drop(columns="date").to_csv("customer_activity.csv", index=False)

# behavior at the scoring date, used for the recommendations
signal_cols = ["revenue_30d", "revenue_60d", "revenue_90d", "active_days_30d", "recharge_count_30d",
               "days_since_last_revenue", "days_since_last_recharge", "avg_recharge_amount",
               "data_revenue_share", "voice_revenue_share"]
features.loc[lookup_ids, signal_cols].rename_axis("customer_id").to_csv("customer_signals.csv")

print(customer_activity.shape, len(lookup_ids))


### Business Recommendations

- Contact the "Retain now" customers (127) within 7 days, through the most direct channel available, starting with the highest expected revenue at risk. They are at high risk and were valuable now or until recently.
- Respond to inactivity step by step rather than only after 90 days. Churn rises from 4% for customers who paid in the last week to 11% after 8 to 14 days, 25% after 15 to 30 days and 40% after 31 to 60 days, but most customers with short gaps come back on their own (91% of gaps of a week or more end with the customer paying again). A free reminder after about two weeks without revenue, a small offer after 30 days and a win-back offer after 60 days avoid spending on customers who would return anyway. Valuable customers can be contacted earlier and more directly, and where possible the trigger should be silence that is unusually long for that customer, since a typical customer recharges about every 5 days.
- Watch for falling spending. Customers who churned kept stable spending until about 60 days before the scoring date and then cut it sharply. Refreshing churn scores every month and flagging customers whose spending fell by 30% or more catches them while they can still be reached.
- Spend little on low-value risk. "Low-cost action" customers (551) should get cheap, automated contact only, such as an SMS with a small recharge or bundle bonus.
- Keep loyal customers loyal. The 1,102 low-risk customers need no retention spend; high-value customers in this group are good candidates for upgrades or loyalty rewards.
- Match offers to how customers spend: data bundles for data-heavy customers and minutes for voice-heavy customers. 1,893 customers used data without data revenue, possibly through bundles bought with airtime (to be confirmed with the data owners).
- Target win-back campaigns at customers who paid before and then went quiet. 28% of them came back on their own, against 9% of customers who never paid.
- Measure the effect of retention campaigns by keeping a small random control group, and compare churn between contacted and not contacted customers.
- Track churn, CLV and customer satisfaction (CSAT) together. Churn and CLV show who leaves and what they are worth, but not why; dissatisfied customers often leave before their spending shows it. A short CSAT survey after customer care contacts and campaigns, linked to customer IDs, would also give the churn model a new predictor.
- Improve the data: link recharges to the bundles they buy (2,298 customers recharged more than their recorded revenue), and add each customer's subscription plan, CSAT, complaints, network quality data and a longer history.


### Limitations and Next Steps

- One scoring date: the churn model was trained and tested at a single cutoff (2026-07-02), so results may differ at other dates.
- Short history: about 13 months of data limits the CLV feature window to 180 days and allows only one out-of-time check.
- Seasonality: the CLV feature window (2025-10-06 to 2026-04-03) includes the December revenue peak, which may make some customers look like bigger spenders than usual.
- Incomplete spending data: 1,893 customers used data without data revenue and 2,298 recharged more than their recorded revenue, so recorded revenue may not show all spending.
- Missing drivers of churn: there is no subscription plan, satisfaction (CSAT), complaints or network quality data, so the models show who is likely to leave but not why.
- CLV model: on amounts, the simple baseline was more accurate on the test set (MAE 2,010 against 2,311) and about equal on the later period. The random forest is used because it ranks customers better (Spearman 0.78 against 0.72), and it underestimates the biggest spenders.
- Expected revenue at risk is an estimate for ranking, not a forecast of lost revenue, because the spending drop before churn is partly counted twice.
- Rules of thumb: the risk cut-offs (10% and 30%), the value split (50/30/20) and the inactivity triggers (14, 30 and 60 days) were chosen for clarity and should be tested in real campaigns.
- Next steps: run retention campaigns with a control group, add the missing data, refresh scores monthly, and retest the models (for example gradient boosting) once a longer history is available.
